# Sensors Analytics REST API

## Deploy the API
- Go to the *Machine* tab, then set *Incoming connections* to **ON**. The API will be accessible through the indicated tunnelling link.  
- Run the notebook.



## Import the required modules

In [1]:
import cherrypy
import json
import redis

## Connect to the Redis Database

In [2]:
REDIS_HOST = 'redis-12335.c74.us-east-1-4.ec2.redns.redis-cloud.com'
REDIS_PORT = 12335
REDIS_USERNAME = 'default'
REDIS_PASSWORD = 'WAZqGdJLk3ZVC1nGmcYggucmetQKGbb1'

redis_client = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, username=REDIS_USERNAME, password=REDIS_PASSWORD)

is_connected = redis_client.ping()
print('Redis Connected:', is_connected)

Redis Connected: True


## Instructions:
1) Create a class for each endpoint (status, sensors, sensor)
    - Status
    - Sensors
    - Sensor
2) For each endpoint, implement the required HTTP methods (GET, POST, PUT, DELETE)
    - Status: GET
    - Sensors: GET, POST
    - Sensor: GET, PUT, DELETE
3) Map each object to its target endpoint.
    - Status() -> "/status"
    - Sensors() -> "/sensors"
    - Sensor() -> "/sensor"

## Status Endpoint Class

In [3]:
class Status(object):
    exposed = True

    def GET(self, *path, **query):
        response_dict = {
            'status': 'online'
        }
        response = json.dumps(response_dict)

        return response

## Sensors Endpoint Class

In [4]:
class Sensors(object):
    exposed = True

    def GET(self, *path, **query):
        # print(query)
        min_t_samples = int(query.get('min_t_samples', 0))
        min_h_samples = int(query.get('min_h_samples', 0))
        sensors = []
        keys = redis_client.keys('0x*:temperature')

        count = 0
        for key in keys:
            key = key.decode()
            mac_address = key.split(':')[0]

            t_info = redis_client.ts().info(f'{mac_address}:temperature')
            t_samples = t_info.total_samples
            t_retention = t_info.retention_msecs
            h_info = redis_client.ts().info(f'{mac_address}:humidity')
            h_samples = h_info.total_samples
            h_retention = h_info.retention_msecs
            
            if t_samples >= min_t_samples and h_samples >= min_h_samples:
                sensors.append(
                    {
                        "mac_address": mac_address,
                        "t_samples": t_samples,
                        "t_retention": t_retention,
                        "h_samples": h_samples,
                        "h_retention": h_retention,
                    }
                )
                count += 1

        response_dict = {
            "sensors": sensors,
            "count": count,
        }

        response = json.dumps(response_dict)

        return response

    def POST(self, *path, **query):
        body = cherrypy.request.body.read()
        # print(body)
        body_dict = json.loads(body.decode())
        # print(body_dict)

        mac_address = body_dict.get('mac_address', None)

        if mac_address is None:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request body.')

        try:
            redis_client.ts().create(f'{mac_address}:temperature', retention_msecs=24*60*60*1000)
        except redis.ResponseError:
            raise cherrypy.HTTPError(409, 'Sensor already exists.')

        try:
            redis_client.ts().create(f'{mac_address}:humidity', retention_msecs=24*60*60*1000)
        except redis.ResponseError:
            raise cherrypy.HTTPError(409, 'Sensor already exists.')

        return

## Sensor Endpoint Class

In [5]:
class Sensor(object):
    exposed = True

    def GET(self, *path, **query):
        if len(path) != 1:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request parameters.')

        mac_address = path[0]

        try:
            t_info = redis_client.ts().info(f'{mac_address}:temperature')
        except redis.ResponseError:
            raise cherrypy.HTTPError(404, 'MAC address not found in the database.')
        
        h_info = redis_client.ts().info(f'{mac_address}:humidity')

        response_dict = {
            "mac_address": mac_address,
            "t_samples": t_info.total_samples,
            "t_retention": t_info.retention_msecs,
            "h_samples": h_info.total_samples,
            "h_retention": h_info.retention_msecs,
        }

        response = json.dumps(response_dict)

        return response

    def PUT(self, *path, **query):
        if len(path) != 1:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request parameters.')

        mac_address = path[0]

        try:
            t_info = redis_client.ts().info(f'{mac_address}:temperature')
        except redis.ResponseError:
            raise cherrypy.HTTPError(404, 'MAC address not found in the database.')

        body = cherrypy.request.body.read()
        # print(body)
        body_dict = json.loads(body.decode())
        # print(body_dict)

        t_retention = body_dict.get('t_retention', None)

        if t_retention is None:
            raise cherrypy.HTTPError(400, 'Missing temperature retention period in the request body.')

        h_retention = body_dict.get('h_retention', None)

        if h_retention is None:
            raise cherrypy.HTTPError(400, 'Missing humidity retention period in the request body.')

        redis_client.ts().alter(f'{mac_address}:temperature', retention_msecs=t_retention)
        redis_client.ts().alter(f'{mac_address}:humidity', retention_msecs=h_retention)

        return
    
    def DELETE(self, *path, **query):
        if len(path) != 1:
            raise cherrypy.HTTPError(400, 'Missing MAC address in the request parameters.')
        
        mac_address = path[0]
        found = 0
        found += redis_client.delete(f'{mac_address}:temperature')
        found += redis_client.delete(f'{mac_address}:humidity')

        if found == 0:
            raise cherrypy.HTTPError(404, 'MAC address not found in the database.')

        return

## Setup cherrypy and Map objects to their target endpoints

In [6]:
if __name__ == '__main__':
    conf = {'/': {'request.dispatch': cherrypy.dispatch.MethodDispatcher()}}
    cherrypy.tree.mount(Status(), '/status', conf)
    cherrypy.tree.mount(Sensors(), '/sensors', conf)
    cherrypy.tree.mount(Sensor(), '/sensor', conf)
    cherrypy.config.update({'server.socket_host': '0.0.0.0'})
    cherrypy.config.update({'server.socket_port': 8080})
    cherrypy.engine.start()
    cherrypy.engine.block()

[05/Dec/2024:14:02:25] ENGINE Bus STARTING
[05/Dec/2024:14:02:25] ENGINE Started monitor thread 'Autoreloader'.
[05/Dec/2024:14:02:25] ENGINE Serving on http://0.0.0.0:8080
[05/Dec/2024:14:02:25] ENGINE Bus STARTED
172.3.28.49 - - [05/Dec/2024:14:02:34] "GET /status HTTP/1.1" 200 20 "" "vscode-restclient"
172.3.186.144 - - [05/Dec/2024:14:03:17] "GET /status HTTP/1.1" 200 20 "" "vscode-restclient"
172.3.50.42 - - [05/Dec/2024:14:03:18] "GET /sensors HTTP/1.1" 200 27 "" "vscode-restclient"
172.3.186.144 - - [05/Dec/2024:14:09:26] "GET /status HTTP/1.1" 200 20 "" "python-requests/2.28.1"
172.3.50.42 - - [05/Dec/2024:14:09:27] "POST /sensors HTTP/1.1" 200 - "" "python-requests/2.28.1"
172.3.50.42 - - [05/Dec/2024:14:09:28] "GET /sensors HTTP/1.1" 200 142 "" "python-requests/2.28.1"
172.3.28.49 - - [05/Dec/2024:14:10:44] "DELETE /sensor/0xdca632c91d8d HTTP/1.1" 200 - "" "vscode-restclient"
172.3.50.42 - - [05/Dec/2024:14:10:58] "GET /status HTTP/1.1" 200 20 "" "python-requests/2.28.1"
172.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=ab5228b8-e3a2-4c2f-9e86-8cb52d8a840d' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>